In [ ]:
####################################################################
# ProjectTwoDashboard.ipynb
# Interactive Dash dashboard for Grazioso Salvare that displays
# Austin Animal Center (AAC) outcome data, lets users filter animals
# by rescue type, and shows results in a data table, pie chart,
# and an interactive Leaflet map.
# Edward McCauley
# CS-340 Client Server Development
####################################################################

# ---------- Imports ----------

# JupyterDash lets us run a Dash web app inside a Jupyter notebook.
from jupyter_dash import JupyterDash

# dash_leaflet provides interactive map components (tiles, markers,popups)
import dash_leaflet as dl

# Core Dash building blocks:
#   dcc  = Dash Core Components (dropdowns, radio buttons, graphs, etc.)
#   html = HTML wrappers (div, h1, p, img, etc.) used to build the page
from dash import dcc, html

# Plotly Express is a high-level charting library. We use it here to
# create a pie chart of breed distribution.
import plotly.express as px

# DataTable is a Dash component that renders an interactive spreadsheet-
# like table with sorting, filtering, and pagination built in
from dash import dash_table

# Input, Output, and State are used in Dash "callbacks"
# functions that run automatically when a user interacts with a component.
#   Output = the component property that will be updated
#   Input  = the component property that triggers the callback
#   State  = extra data read when the callback fires (does not trigger)
from dash.dependencies import Input, Output, State

# base64 encodes binary data (like an image file) into a text string
# so it can be embedded directly in HTML as a data URI.
import base64

# Detect the Jupyter proxy configuration so the Dash server URL
# works correctly inside hosted notebook environments like Codio.
JupyterDash.infer_jupyter_proxy_config()

# pandas is a Python library for tabular data. DataFrames
# (2-D tables with labeled columns) make it easy to manipulate,
# filter, and display data from MongoDB.
import pandas as pd

# Import the CRUD helper class in CRUD_Python_Module6.py.
# This keeps all MongoDB logic (connect, insert, find, update, delete)
# in one reusable module so the dashboard code stays clean.
from CRUD_Python_Module6 import AnimalShelter


##################################################################
# Data Manipulation / Model
##################################################################
# This section connects to MongoDB, defines the rescue-type filter
# queries, and loads the initial dataset into a pandas DataFrame.
##################################################################

# Database credentials
username = "aacuser"
password = "aacuser"

# Create an instance of our CRUD class. The constructor opens a
# connection to MongoDB using the credentials above.
db = AnimalShelter(username, password)

# ---------- Rescue-Type Filter Queries ----------
# Each query is a MongoDB filter dictionary. When the user selects a
# rescue type, the corresponding query is sent to db.read() to fetch
# only the animals that match.
#
# MongoDB query operators used:
#   $in  : matches if the field value is in the given list
#   $gte : greater than or equal to
#   $lte : less than or equal to

# Water Rescue dogs: specific breeds, intact females, 26-156 weeks old
WATER_QUERY = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "Labrador Retriever Mix", "Chesapeake Bay Retriever",
        "Newfoundland",
    ]},
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}

# Mountain / Wilderness Rescue: specific breeds, intact males, 26-156 weeks
MOUNTAIN_QUERY = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "German Shepherd", "Alaskan Malamute",
        "Old English Sheepdog", "Siberian Husky", "Rottweiler",
    ]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}

# Disaster / Individual Tracking: specific breeds, intact males, 20-300 weeks
DISASTER_QUERY = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "Doberman Pinscher", "German Shepherd",
        "Golden Retriever", "Bloodhound", "Rottweiler",
    ]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
}

# ---------- Initial Data Load ----------
# db.read({}) with an empty dict fetches ALL documents.
# pd.DataFrame.from_records() converts the list of dicts into a table.
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if "_id" in df.columns:
    df.drop(columns=['_id'],inplace=True)


####################################################################
# Dashboard Layout / View
####################################################################
# The layout defines the visual structure of the page using nested
# HTML and Dash components.
####################################################################

# Create the Dash app instance. __name__ helps Dash find static assets.
app = JupyterDash(__name__)

# Load the Grazioso Salvare’s company logo from disk and base64-encode it 
# so we can embed it directly in the HTML without needing a separate 
# web server to serve the image file.
# "with open(...) as f" is a context manager -- it guarantees the file
# is closed automatically when the block ends, even if an error occurs.
image_filename = 'Grazioso Salvare Logo.png' 
with open(image_filename, 'rb') as f:
    encoded_image = base64.b64encode(f.read())
    
# app.layout is the root of the component tree. Every visible element
# on the page is a child of this outer html.Div.
app.layout = html.Div([
    
    # Hidden div is a common Dash pattern used as a placeholder for
    # callbacks that need an Output but don't visually display anything.
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Page title
    html.Center(html.B(html.H1('Grazioso Salvare: Austin Animal Center Outcomes Dashboard'))),
    
    # Embed the base64-encoded logo as an <img> tag.
    # .decode() converts bytes to a UTF-8 string for the src attribute.
    html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'height': '100px'},
    ),
    
    html.Br(),
    
    # URL anchor tag to the client's home page, displayed as a
    # clickable text link below the logo. html.A creates an <a> tag;
    # target="_blank" opens the link in a new browser tab.
    html.A(
        "www.snhu.edu",
        href="https://www.snhu.edu",
        target="_blank",
    ),

    html.H3("CS-340 - Project 2 - emccauley707502143"),

    
    html.Hr(),
    
    # ---------- Radio Buttons for Rescue-Type Filter ----------
    # dcc.RadioItems creates a set of radio buttons. When the user
    html.Div(
            children=[
                html.H3("Interactive Filter Options"),
                dcc.RadioItems(
                    id="filter-type",                  # Unique ID used by callbacks
                    options=[
                        {"label": "Water Rescue", "value": "water"},
                        {"label": "Mountain / Wilderness Rescue", "value": "mountain"},
                        {"label": "Disaster / Individual Tracking", "value": "disaster"},
                        {"label": "Reset", "value": "reset"},
                    ],
                    value="reset",                     # default selection on page load
                    inline=True,                       # horizontal displayed buttons
                ),
            ],
            style={"padding": "10px"},
        ),
    
    
    html.Hr(),
    
    # ---------- Interactive Data Table ----------
    # dash_table.DataTable renders the animal data as a sortable,
    # filterable, paginated table. Many of its properties are
    # configurable. https://dash.plotly.com/datatable    
    dash_table.DataTable(
        id='datatable-id',
        
        # Build columns dynamically from the DataFrame
        # Each dict needs "name" (display header) and "id" (data key).
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} 
            for i in df.columns
        ],
        
        # Convert DataTable to a list of dictionaries
        data=df.to_dict('records'),
        
        # Prevent editing/deleting rows
        editable=False,
        row_deletable=False,
        
        # Dash Native UX features (https://dash.plotly.com/datatable/interactivity)
        filter_action="native",      # Filter by column
        sort_action="native",        # Sort by column headers
        sort_mode="multi",           # Multi-column sorting
        page_action="native",        # Pagination in the browser
        page_current=0,              # Start on the first page
        page_size=10,                # Customized 10 rows per page
        
        # Column selection is off. Use row selection for the map.
        column_selectable=False,     
        selected_columns=[],
        
        # Leaflet map callback
        row_selectable="single",     # Requirement: single-row selection
        selected_rows=[0],           # First row selected by default
        
        # Exploring some of the stylistic options to make your data table more user-friendly
        # Style and easier to use
        style_table={
        "overflowX": "auto",         # horizontal scroll if many columns
        "maxHeight": "450px",        # keeps the dashboard compact
        "overflowY": "auto",
        },
        style_cell={
        "textAlign": "left",
        "padding": "8px",
        "whiteSpace": "normal",      # wrap text
        "height": "auto",
        "minWidth": "90px",
        "maxWidth": "300px",
        },
        # Bold header font
        style_header={
            "fontWeight": "bold",
            "backgroundColor": "rgb(220, 220, 220)",
        },
        style_data_conditional=[
            # Striped rows for readability (https://dash.plotly.com/datatable/style)
            {"if": {"row_index": "odd"}, 
             "backgroundColor": "rgb(220, 220, 220)"},

            # Highlight the selected row so users know what the map is showing
            {"if": {"state": "selected"},
             "backgroundColor": "#D2F3FF"},
        ],
     ),

    html.Br(),
    html.Hr(),
    
    # ---------- Chart + Map Side by Side ----------
    # This sets up the dashboard so that your chart 
    # and your geolocation chart are side-by-side. 
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),    # Pie chart
            html.Div(id='map-id', className='col s12 m6'),      # Map
        ]
    ),
])


####################################################################
# Interaction Between Components / Controller
###################################################################
# Callbacks connect the user actions with the visual updates.
# Dash watches each Input; when its value changes, Dash calls the
# decorated function and pushes the return value into each Output.
####################################################################
    
# ---------- Callback 1: Filter the Data Table ----------
# Triggered when the user clicks a radio button (filter-type value changes).
# Outputs: new table data and a reset of the selected row.
@app.callback(
    Output("datatable-id", "data"),            # table rows to display
    Output("datatable-id", "selected_rows"),   # which row is highlighted
    Input("filter-type", "value"),             # the radio button selection
)
def update_table(filter_type):
    """Query MongoDB for animals matching the selected rescue type
    and refresh the DataTable with the results."""

    # Pick the right query dict based on the radio button value.
    if filter_type == "water":
        query = WATER_QUERY
    elif filter_type == "mountain":
        query = MOUNTAIN_QUERY
    elif filter_type == "disaster":
        query = DISASTER_QUERY
    else:
        # "reset" or any unknown value -> fetch all animals
        query = {}

    # Run the query through our CRUD module and build a new DataFrame.
    dff = pd.DataFrame.from_records(db.read(query))

    # Remove the _id column again (same reason as the initial load).
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    # Return the new data as list[dict] and pre-select the first row
    # so the map callback always has a valid row to render.
    return dff.to_dict("records"), [0]

# ---------- Callback 2: Update the Pie Chart ----------
# Display the breeds of animal based on quantity represented in the data table.
# Triggered when: the visible table data changes (after filtering,
# sorting, or paging). "derived_virtual_data" is the data currently
# shown on the active page of the table.
@app.callback(
    Output("graph-id", "children"),              # inject chart into this div
    Input("datatable-id", "derived_virtual_data"),
)
def update_graph(viewData):
    """Build a pie chart showing the breed distributions in the
    currently visible table data."""

    # If the table is empty (no matching animals), show a message.
    if not viewData:
        return html.Div("No data for chart.")

    # Convert the list of dicts back into a DataFrame for easy analysis.
    dff = pd.DataFrame.from_records(viewData)

    # Safety check in case the "breed" column is missing.
    if "breed" not in dff.columns:
        return html.Div("No breed column found.")

    # value_counts() counts occurrences of each breed and sorts descending.
    counts = dff["breed"].value_counts()

    # If more than 10 breeds, keep the top 10 individually, and summ all
    # remaining breeds into a single "Other" category. This prevents the
    # pie chart from becoming cluttered and unreadable when the unfiltered
    # (Reset) dataset contains hundreds of breeds.
    if len(counts) > 10:
        top = counts.head(10)
        other_count = counts.iloc[10:].sum()
        top["Other"] = other_count
        counts = top
    
    # Build the DataFrame explicitly with named columns.
    chart_df = pd.DataFrame({"breed": counts.index, "count": counts.values})

    # px.pie() creates an interactive Plotly pie chart.
    fig = px.pie(chart_df, names="breed", values="count",
                 title="Breed Distributions")

    # Wrap the figure in a dcc.Graph so Dash can render it.
    return dcc.Graph(figure=fig)
    
# ---------- Callback 3: Update the Leaflet Map ----------
# This callback will highlight a cell on the data table when the user selects it
# Triggered when: the visible table data changes OR the user selects
# a different row in the table.
@app.callback(
    Output('map-id', "children"),
    Input('datatable-id', "derived_virtual_data"),
    Input('datatable-id', "derived_virtual_selected_rows"),
)
def update_map(viewData, index):
    """Place a marker on the Leaflet map at the geo-location of the
    selected animal, with a tooltip showing breed and a popup showing
    the animal's name."""

    # If there is no data at all, show a placeholder message.
    if viewData is None or not viewData:
        return html.Div("No data for map.")

    # Determine which row the user selected. If nothing is selected,
    # default to the first row (index 0).
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Convert the list of dicts into a DataFrame for iloc access.
    dff = pd.DataFrame.from_dict(viewData)

    # Bounds check: if the selected index exceeds the number of rows
    # (e.g., after a filter reduced the data), fall back to row 0.
    if row >= len(dff):
        row = 0
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],    # Austin, TX coordinates
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"],
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row]["name"]),
                        ]),
                    ],
                ),
            ],
        )
    ]


# ---------- Start the Server ----------
# This launches the Dash development server. Run app and display result in jupyterlab mode, 
# note, if you have previously run a prior app, the default port of 8050 may not be available, 
# if so, try setting an alternate port.
app.run_server() 